# Model Debugging: Comparing TranslatedPublishedModel vs FullyTranslatedPublishedModel

This notebook compares the two model versions to identify why the fully translated model is not producing biomass.

In [7]:
%run util.py
import json
import pandas as pd
import cobra
from cobra.io import load_json_model

/home/chenry/projects/KBUtilLib/src
modelseedpy 0.4.2


2025-10-31 16:30:03,315 - __main__.NotebookUtil - WARNING - Section 'ModelSEEDBiochem' not found in config


cobrakbase 0.4.0


## Load Both Models

In [8]:
# Load the working model (TranslatedPublishedModel)
working_model = load_json_model('models/TranslatedPublishedModel.json')
print(f"Working model: {working_model.id}")
print(f"Number of reactions: {len(working_model.reactions)}")
print(f"Number of metabolites: {len(working_model.metabolites)}")
print(f"Number of genes: {len(working_model.genes)}")

Working model: iAbaylyiv4
Number of reactions: 990
Number of metabolites: 828
Number of genes: 776


In [9]:
# Load the non-working model (FullyTranslatedPublishedModel)
broken_model = load_json_model('models/FullyTranslatedPublishedModel.json')
print(f"Broken model: {broken_model.id}")
print(f"Number of reactions: {len(broken_model.reactions)}")
print(f"Number of metabolites: {len(broken_model.metabolites)}")
print(f"Number of genes: {len(broken_model.genes)}")

Broken model: iAbaylyiv4
Number of reactions: 990
Number of metabolites: 828
Number of genes: 776


## Test Biomass Production

In [16]:
%run util.py
from cobra.io import save_json_model, load_json_model
pubmod = MSModelUtil.from_cobrapy("models/TranslatedPublishedModel.json")
pubmod.model.reactions.get_by_id("rxn01332_c0").lower_bound = 0
pubmod.model.reactions.get_by_id("rxn01332_c0").upper_bound = 0
pyruvate_media = util.get_media("KBaseMedia/Carbon-Pyruvic-Acid")
growth_solution = util.run_fba(pubmod,media=pyruvate_media,objective="GROWTH_DASH_RXN",run_pfba=True)
growth_solution

2025-10-31 16:39:18,584 - __main__.NotebookUtil - WARNING - Section 'ModelSEEDBiochem' not found in config


/home/chenry/projects/KBUtilLib/src


INFO:modelseedpy.core.msmodelutl:cpd00063 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd10516 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00205 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00254 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00099 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00058 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00030 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00034 not found in model!


,fluxes,reduced_costs
rxn12357_c0,-0.005348,2.000000
rxn00947_c0,-0.009022,2.000000
rxn00100_c0,0.001875,-2.000000
rxn00305_c0,0.000000,-1.000000
rxn04783_c0,-0.297079,2.000000
...,...,...
rxn01477_c0,0.000000,-2.000000
EXF_DASH_DEOXYCYTIDINE_LBRACKET_Extraorganism_RBRACKET_,0.000000,270.166667
rxn01519_c0,0.013804,-2.000000
rxn05667_c0,0.000000,-2.000000


In [20]:
%run util.py
from util_legacy import NotebookUtil
_legacy = NotebookUtil()
# Test growth in working model
try:
    model = MSModelUtil.from_cobrapy("models/TranslatedPublishedModel.json")
    model.model.reactions.get_by_id("rxn01332_c0").lower_bound = 0
    model.model.reactions.get_by_id("rxn01332_c0").upper_bound = 0
    pyruvate_media = _legacy.get_media("KBaseMedia/Carbon-Pyruvic-Acid")
    solution_working = _legacy.run_fba(model,media=pyruvate_media,objective="GROWTH_DASH_RXN",run_pfba=True)
    print(f"Working model growth rate: {solution_working.objective_value}")
    print(f"Status: {solution_working.status}")
except Exception as e:
    print(f"Error optimizing working model: {e}")

INFO:modelseedpy.core.msmodelutl:cpd00063 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd10516 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00205 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00254 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00099 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00058 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00030 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00034 not found in model!


Working model growth rate: 0.44686403216160103
Status: optimal


In [22]:
%run util.py
from util_legacy import NotebookUtil
_legacy = NotebookUtil()
# Test growth in broken model
try:
    model = MSModelUtil.from_cobrapy("models/FullyTranslatedPublishedModel.json")
    model.model.reactions.get_by_id("rxn01332_c0").lower_bound = 0
    model.model.reactions.get_by_id("rxn01332_c0").upper_bound = 0
    solution_broken = _legacy.run_fba(model,media="KBaseMedia/Carbon-Pyruvic-Acid",objective="GROWTH_DASH_RXN",run_pfba=True)
    print(f"Broken model growth rate: {solution_broken.objective_value}")
    print(f"Status: {solution_broken.status}")
except Exception as e:
    print(f"Error optimizing broken model: {e}")

INFO:modelseedpy.core.msmodelutl:cpd00063 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd10516 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00205 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00254 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00099 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00058 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00030 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00034 not found in model!


Broken model growth rate: 0.44686403216160103
Status: optimal


## Compare Reactions

In [ ]:
# Get reaction IDs from both models
working_rxns = set(r.id for r in working_model.reactions)
broken_rxns = set(r.id for r in broken_model.reactions)

# Find differences
only_in_working = working_rxns - broken_rxns
only_in_broken = broken_rxns - working_rxns
common_rxns = working_rxns & broken_rxns

print(f"Reactions only in working model: {len(only_in_working)}")
print(f"Reactions only in broken model: {len(only_in_broken)}")
print(f"Common reactions: {len(common_rxns)}")

In [ ]:
# Show reactions only in working model
if only_in_working:
    print("\nReactions ONLY in working model:")
    for rxn_id in sorted(only_in_working):
        rxn = working_model.reactions.get_by_id(rxn_id)
        print(f"  {rxn_id}: {rxn.reaction}")

In [ ]:
# Show reactions only in broken model
if only_in_broken:
    print("\nReactions ONLY in broken model:")
    for rxn_id in sorted(only_in_broken):
        rxn = broken_model.reactions.get_by_id(rxn_id)
        print(f"  {rxn_id}: {rxn.reaction}")

## Compare Flux Bounds for Common Reactions

In [ ]:
# Compare bounds for all common reactions
bound_differences = []

for rxn_id in common_rxns:
    rxn_working = working_model.reactions.get_by_id(rxn_id)
    rxn_broken = broken_model.reactions.get_by_id(rxn_id)
    
    if (rxn_working.lower_bound != rxn_broken.lower_bound or 
        rxn_working.upper_bound != rxn_broken.upper_bound):
        bound_differences.append({
            'reaction_id': rxn_id,
            'working_lb': rxn_working.lower_bound,
            'working_ub': rxn_working.upper_bound,
            'broken_lb': rxn_broken.lower_bound,
            'broken_ub': rxn_broken.upper_bound,
            'reaction': rxn_working.reaction
        })

print(f"\nFound {len(bound_differences)} reactions with different bounds")

In [ ]:
# Display bound differences as a DataFrame
if bound_differences:
    df_bounds = pd.DataFrame(bound_differences)
    print("\nReactions with different flux bounds:")
    display(df_bounds)
else:
    print("\nNo differences in flux bounds found.")

## Compare Gene Associations

In [ ]:
# Compare gene associations for common reactions
gpr_differences = []

for rxn_id in common_rxns:
    rxn_working = working_model.reactions.get_by_id(rxn_id)
    rxn_broken = broken_model.reactions.get_by_id(rxn_id)
    
    gpr_working = str(rxn_working.gene_reaction_rule)
    gpr_broken = str(rxn_broken.gene_reaction_rule)
    
    if gpr_working != gpr_broken:
        gpr_differences.append({
            'reaction_id': rxn_id,
            'working_gpr': gpr_working,
            'broken_gpr': gpr_broken,
            'reaction': rxn_working.reaction
        })

print(f"\nFound {len(gpr_differences)} reactions with different GPRs")

In [ ]:
# Display GPR differences
if gpr_differences:
    df_gpr = pd.DataFrame(gpr_differences)
    print("\nReactions with different gene associations:")
    display(df_gpr.head(20))  # Show first 20
    print(f"\nTotal GPR differences: {len(gpr_differences)}")
else:
    print("\nNo differences in gene associations found.")

## Check Biomass Reaction Specifically

In [ ]:
# Find and compare biomass reactions
print("Working model objective:")
print(f"  {working_model.objective}")
working_biomass = working_model.reactions.get_by_id(list(working_model.objective.variables.keys())[0].id)
print(f"  Bounds: [{working_biomass.lower_bound}, {working_biomass.upper_bound}]")
print(f"  Reaction: {working_biomass.reaction}")

print("\nBroken model objective:")
print(f"  {broken_model.objective}")
broken_biomass = broken_model.reactions.get_by_id(list(broken_model.objective.variables.keys())[0].id)
print(f"  Bounds: [{broken_biomass.lower_bound}, {broken_biomass.upper_bound}]")
print(f"  Reaction: {broken_biomass.reaction}")

## Export Detailed Comparison

In [ ]:
# Save detailed comparison to Excel file
with pd.ExcelWriter('nboutput/model_comparison.xlsx') as writer:
    # Bound differences
    if bound_differences:
        df_bounds.to_excel(writer, sheet_name='Bound_Differences', index=False)
    
    # GPR differences
    if gpr_differences:
        df_gpr.to_excel(writer, sheet_name='GPR_Differences', index=False)
    
    # Reaction differences
    if only_in_working or only_in_broken:
        rxn_diff_data = []
        for rxn_id in sorted(only_in_working):
            rxn = working_model.reactions.get_by_id(rxn_id)
            rxn_diff_data.append({
                'reaction_id': rxn_id,
                'model': 'working_only',
                'reaction': rxn.reaction,
                'lower_bound': rxn.lower_bound,
                'upper_bound': rxn.upper_bound
            })
        for rxn_id in sorted(only_in_broken):
            rxn = broken_model.reactions.get_by_id(rxn_id)
            rxn_diff_data.append({
                'reaction_id': rxn_id,
                'model': 'broken_only',
                'reaction': rxn.reaction,
                'lower_bound': rxn.lower_bound,
                'upper_bound': rxn.upper_bound
            })
        pd.DataFrame(rxn_diff_data).to_excel(writer, sheet_name='Reaction_Differences', index=False)

print("\nComparison saved to nboutput/model_comparison.xlsx")